# Intent Classification and Slot Filling

Imagine you are building a virtual assistant for flight booking. A user says:
*"I need a flight from Rome to Paris tomorrow morning."*

The system must understand both what the user is asking for (booking a flight) and extract relevant details such as departure city, destination, and date.

How can a model understand the user's request and extract the relevant information from the sentence?

### Objectives
- How can we model intent classification and slot filling with Transformers?
- How can GPT2 be adapted to perform sequence labeling and classification?
- Should our model be autoregressive?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/massimo-rizzoli/NLU-2026-Labs/blob/main/labs/05_intent_and_slot_filling.ipynb)

# 1 Sequence Labeling,  Shallow Parsing and Text classification tasks

## 1.1 Sequence Labeling and Shallow parsing
Sequence labelling is to assign a label for each token. The task is formally defined as:
- Given a sequence of tokens $w = {w_1, w_2, ..., w_n}$,
- defining a sequence of labels as $l = {l_1, l_2, ..., l_n}$
- compute the sequence $\hat{l}$ such as $\hat{l} = \underset{l}{\operatorname{argmax}} P(l|w)$ 

A particular case of sequence labelling is [Shallow Parsing](https://en.wikipedia.org/wiki/Shallow_parsing). The main difference from Sequence Labeling task is that Shallow Parsing performs **segmentation** of input sequence into constituents. Segmentation is required to identify categories (or types) of *multi-word expressions*.

In this, we are going to see a particular case of shallow parsing task, which is named as Slot Filling (or Concept tagging). The **segmentation** part is represented with IOB tags and the **labeling** part are the concepts defined in the annotation schema of a corpus. \
\
An example is the following: 

| Slot Filling |  |                     |                     |  |  |  |  |
|------------------|----|--------------------------|--------------------------|---|------|---|--------|
| Input sequence:  | on | april                    | first                    | I | want | a | flight |
| Output sequence: | O  | B-depart_date.month_name | B-depart_date.day_number | O | O    | O | O      |

Another example:

| Slot Filling |  |                     |                     |  |  |  |  | | |
|------------------|----|--------------------------|--------------------------|---|------|---|--------|---|---|
| Input sequence:  | I  | want                     | a                        | flight | to | New | York| next | Sunday |
| Output sequence: | O  | O                        |                          | O | O    | B-toloc.city_name | I-toloc.city_name      | B-arrive_date.date_relative | B-arrive_date.day_name|

## 1.2 Text classification
The text classification problem is defined  as follows:
- Given a sequence of tokens $w = {w_1, w_2, ..., w_n}$,
- And a set of labels $L$ where $l \in L$
- estimate the label $\hat{l}$ such as $\hat{l} = \underset{l}{\operatorname{argmax}} P(l|w)$ 

In text classification, the label is given to the whole input sequence instead of at each element of the sequence (as in sequence labelling).

The text classification task that we are going to see in this laboratory is named as Intent Classification. The Intent is an additional component of the *semantic frame*. \
\
An example is the following:

| Intent Classification|  |                     |                     |  |  |  |  |
|------------------|----|--------------------------|--------------------------|---|------|---|--------|
| Input sequence:  | on | april                    | first                    | I | want | a | flight |
| Output label: | flight     |


# 2 Dataset
The dataset that we are going to use is ATIS (Airline Travel Information Systems). It is composed of transcriptions of humans asking about flight information.  

## 2.2 Load the dataset
We have a custom data structure for this dataset. The structure is the following:
```json
[
    {
    "utterance": "on april first i need a flight going from phoenix to san diego", 
    "slots": "O B-depart_date.month_name B-depart_date.day_number O O O O O O B-fromloc.city_name O B-toloc.city_name I-toloc.city_name", 
    "intent": "flight"
    },
    "..."
 ]
```

In [ ]:
# If you are using Colab, run this line, then restart the runtime
#%pip install transformers==4.38.0

In [ ]:
# If you are using Colab, run these commands
# !wget -P dataset/ATIS https://raw.githubusercontent.com/massimo-rizzoli/NLU-2026-Labs/main/labs/dataset/ATIS/test.json
# !wget -P dataset/ATIS https://raw.githubusercontent.com/massimo-rizzoli/NLU-2026-Labs/main/labs/dataset/ATIS/train.json
# !wget https://raw.githubusercontent.com/massimo-rizzoli/NLU-2026-Labs/main/labs/conll.py

In [ ]:
# Global variables
import os
DEVICE = 'cuda:0' # cuda:0 means we are using the GPU with id 0, if you have multiple GPU
os.environ['CUDA_LAUNCH_BLOCKING'] = "1" # Used to report errors on CUDA side
PAD_TOKEN = 0

In [ ]:
import json
from pprint import pprint

def load_data(path):
    '''
        input: path/to/data
        output: json 
    '''
    dataset = []
    with open(path) as f:
        dataset = json.loads(f.read())
    return dataset

tmp_train_raw = load_data(os.path.join('dataset','ATIS','train.json'))
test_raw = load_data(os.path.join('dataset','ATIS','test.json'))
print('Train samples:', len(tmp_train_raw))
print('Test samples:', len(test_raw))

pprint(tmp_train_raw[0])

## 2.3 Create a dev set
In the original split the development set (dev set) is missing. To train and find the best hyperparameter of our network the dev set is fundamental. Thus, we have to create it starting from the **traning** set. The dev set is usually the 10% of the dataset. \
Possible sampling strategies:
* Take the last n elements of the training set.
* Do a random sampling from the training set.
* Do a stratified sampling from the training set using one or more criteria. (The best way)
    * For further details look [here](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

In [ ]:
import random
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter

# First we get the 10% of the training set, then we compute the percentage of these examples 

portion = 0.10

intents = [x['intent'] for x in tmp_train_raw] # We stratify on intents
count_y = Counter(intents)

labels = []
inputs = []
mini_train = []

for id_y, y in enumerate(intents):
    if count_y[y] > 1: # If some intents occurs only once, we put them in training
        inputs.append(tmp_train_raw[id_y])
        labels.append(y)
    else:
        mini_train.append(tmp_train_raw[id_y])
# Random Stratify
X_train, X_dev, y_train, y_dev = train_test_split(inputs, labels, test_size=portion, 
                                                    random_state=42, 
                                                    shuffle=True,
                                                    stratify=labels)
X_train.extend(mini_train)
train_raw = X_train
dev_raw = X_dev

y_test = [x['intent'] for x in test_raw]

# Intent distributions
print('Train:')
pprint({k:round(v/len(y_train),3)*100 for k, v in sorted(Counter(y_train).items())})
print('Dev:'), 
pprint({k:round(v/len(y_dev),3)*100 for k, v in sorted(Counter(y_dev).items())})
print('Test:') 
pprint({k:round(v/len(y_test),3)*100 for k, v in sorted(Counter(y_test).items())})
print('='*89)
# Dataset size
print('TRAIN size:', len(train_raw))
print('DEV size:', len(dev_raw))
print('TEST size:', len(test_raw))

## 2.3 Convert words to numbers (word2id)
Neural Networks work with numbers and vectors.
We need to create a dictionary that maps the words and labels in the training set to unique integers $\geq$ 0, called indexes. That is:
- One dictionary for mapping words to ids (w2id)
- One dictionary for mapping slot labels to ids (slot2id)
- One dictionary for mapping intent labels to ids (intent2id)

*We also need to add special tokens such as "pad" and "unk"*

***Example:***

With w2id map the sentence in `sent` into the computed indexes.
```python
dictionary = {"from": 2, "Boston":88, "to":105, "Tokyo":42}
sent = "from Boston to Tokyo" 
# Output:
[2,88,105,42]
```

We will see later how to convert these indexes into vectors (aka embeddings).



In [ ]:
from collections import Counter

w2id = {'pad':PAD_TOKEN, 'unk': 1}
slot2id = {'pad':PAD_TOKEN}
intent2id = {}
# Map the words only from the train set
# Map slot and intent labels of train, dev and test set. 'unk' is not needed.
for example in train_raw:
    for w in example['utterance'].split():
        if w not in w2id:
            w2id[w] = len(w2id)   
    for slot in example['slots'].split():
        if slot not in slot2id:
            slot2id[slot] = len(slot2id)
    if example['intent'] not in intent2id:
        intent2id[example['intent']] = len(intent2id)
        
for example in dev_raw:
    for slot in example['slots'].split():
        if slot not in slot2id:
            slot2id[slot] = len(slot2id)
    if example['intent'] not in intent2id:
        intent2id[example['intent']] = len(intent2id)
        
for example in test_raw:
    for slot in example['slots'].split():
        if slot not in slot2id:
            slot2id[slot] = len(slot2id)
    if example['intent'] not in intent2id:
        intent2id[example['intent']] = len(intent2id)

sent = 'I wanna a flight from Toronto to Kuala Lumpur'
mapping = [w2id[w] if w in w2id else w2id['unk'] for w in sent.split()]

print(mapping)

print('# Vocab:', len(w2id)-2) # we remove pad and unk from the count
print('# Slots:', len(slot2id)-1)
print('# Intent:', len(intent2id))

## 2.4 Lang class
Later we will need to convert those numbers in the original form, so we need to invert those dictionaries. We create a class named as Lang just for convenience.

In [ ]:
from collections import Counter
class Lang():
    def __init__(self, words, intents, slots, cutoff=0, cls=True):
        self.word2id = self.w2id(words, cutoff=cutoff, unk=True, cls=cls)
        self.slot2id = self.lab2id(slots, cls=cls)
        self.intent2id = self.lab2id(intents, pad=False, cls=False)
        self.id2word = {v:k for k, v in self.word2id.items()}
        # cls will have the same id as the pad token
        self.id2slot = {v:k for k, v in self.slot2id.items() if not cls or k != 'cls'}
        self.id2intent = {v:k for k, v in self.intent2id.items()}
        
    def w2id(self, elements, cutoff=None, unk=True, cls=True):
        vocab = {'pad': PAD_TOKEN}
        if unk:
            vocab['unk'] = len(vocab)
        if cls:
            vocab['cls'] = len(vocab)
        count = Counter(elements)
        for k, v in count.items():
            if v > cutoff:
                vocab[k] = len(vocab)
        return vocab
    
    def lab2id(self, elements, pad=True, cls=True):
        vocab = {}
        if pad:
            vocab['pad'] = PAD_TOKEN
        for elem in elements:
            vocab[elem] = len(vocab)
        if cls:
            # when predicting the slots, we want to ignore the CLS
            # CLS will only be used for intent classification
            vocab['cls'] = PAD_TOKEN
        return vocab

In [ ]:
# No set() since we want to compute the cutoff
words = sum([x['utterance'].split() for x in train_raw], []) # sum(list[list], []) -> from list of list to list

# We do not want unk labels (slots), 
# however this depends on the research purpose
corpus = train_raw + dev_raw + test_raw 

slots = set(sum([line['slots'].split() for line in corpus],[]))
intents = set([line['intent'] for line in corpus])

# words are only from te training set
# labels from the whole corpus (we do not want unk labels)
lang = Lang(words, intents, slots, cutoff=0)

## 2.5 Customize the Dataset class
In Pytorch the Dataset class helps you in handeling the dataset. The mandatory methods are ```__init__, __len__ and __getitem__```. <br>
You can find more details here: https://pytorch.org/tutorials/beginner/basics/data_tutorial.html 

In [ ]:
import torch
import torch.utils.data as data

class IntentsAndSlots(data.Dataset):
    # Mandatory methods are __init__, __len__ and __getitem__
    def __init__(self, dataset, lang, unk='unk', cls='cls', add_cls=True):
        self.utterances = []
        self.intents = []
        self.slots = []
        self.unk = unk
        self.cls = cls
        self.add_cls = add_cls
        
        for x in dataset:
            self.utterances.append(x['utterance'])
            self.slots.append(x['slots'])
            self.intents.append(x['intent'])

        self.utt_ids = self.mapping_seq(self.utterances, lang.word2id)
        self.slot_ids = self.mapping_seq(self.slots, lang.slot2id)
        self.intent_ids = self.mapping_lab(self.intents, lang.intent2id)

    def __len__(self):
        return len(self.utterances)

    def __getitem__(self, idx):
        utt = torch.Tensor(self.utt_ids[idx])
        slots = torch.Tensor(self.slot_ids[idx])
        intent = self.intent_ids[idx]
        sample = {'utterance': utt, 'slots': slots, 'intent': intent}
        return sample
    
    # Auxiliary methods
    
    def mapping_lab(self, data, mapper):
        return [mapper[x] if x in mapper else mapper[self.unk] for x in data]
    
    def mapping_seq(self, data, mapper): # Map sequences to number
        res = []
        for seq in data:
            tmp_seq = []
            for x in seq.split():
                if x in mapper:
                    tmp_seq.append(mapper[x])
                else:
                    tmp_seq.append(mapper[self.unk])
            if self.add_cls:
                tmp_seq.append(mapper[self.cls])
            res.append(tmp_seq)
        return res

In [ ]:
# Create our datasets
train_dataset = IntentsAndSlots(train_raw, lang)
dev_dataset = IntentsAndSlots(dev_raw, lang)
test_dataset = IntentsAndSlots(test_raw, lang)

# 3 Batches
Batches are used to handle large datasets in the memory. Since the whole dataset cannot fit in GPU memories, we randomly shuffle the dataset and we split it in small batches that will be processed one at a time.
## 3.1 Padding
Padding is a strategy to fit sequences of different lengths into a matrix. For instance:

| Right padding|   |    |   |  |  |  |  
|---|----|---|---|---|------|---|
| I | saw| a | unk | with | a | telescope | 
| book | me | a | flight | [pad] | [pad] | [pad] | 

| Left padding|   |    |   |  |  |  |  
|---|----|---|---|---|------|---|
| I | saw| a | unk | with | a | telescope | 
| [pad] | [pad] | [pad] | book | me | a | flight | 



### Dataloader
To split the dataset into batches and add padding we will use the DataLoader class. 
```python
DataLoader(Dataset, batch_size=N, collate_fn={custom function}, shuffle=True)
```
*collate_fn* is used to shape the output batch.

In [ ]:
from torch.utils.data import DataLoader

def collate_fn(data):
    def merge(sequences):
        '''
        merge from batch * sent_len to batch * max_len 
        '''
        lengths = [len(seq) for seq in sequences]
        max_len = 1 if max(lengths)==0 else max(lengths)
        # Pad token is zero in our case
        # So we create a matrix full of PAD_TOKEN (i.e. 0) with the shape 
        # batch_size * maximum length of a sequence
        padded_seqs = torch.LongTensor(len(sequences),max_len).fill_(PAD_TOKEN)
        for i, seq in enumerate(sequences):
            end = lengths[i]
            padded_seqs[i, :end] = seq # We copy each sequence into the matrix
        return padded_seqs, lengths

    data_by_key = {}
    for key in data[0].keys():
        data_by_key[key] = [d[key] for d in data]
        
    # We just need one length for packed pad seq, since len(utt) == len(slots)
    src_utt, _ = merge(data_by_key['utterance'])
    y_slots, y_lengths = merge(data_by_key["slots"])
    intent = torch.LongTensor(data_by_key["intent"])
    
    src_utt = src_utt.to(DEVICE) # We load the Tensor on our selected device
    y_slots = y_slots.to(DEVICE)
    intent = intent.to(DEVICE)
    y_lengths = torch.LongTensor(y_lengths).to(DEVICE)
    
    new_item = {}
    new_item["utterances"] = src_utt
    new_item["intents"] = intent
    new_item["y_slots"] = y_slots
    new_item["slots_len"] = y_lengths
    return new_item

# Dataloader instantiations
train_loader = DataLoader(train_dataset, batch_size=128, collate_fn=collate_fn,  shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=64, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=64, collate_fn=collate_fn)

# 4 Model - GPT2
We can use the GPT2 implementation from Part 1 of the project. However, we need to modify the output layer to perform intent classification and slot filling.


<p align="left">
    <img src="https://i.postimg.cc/L64C7HK6/Transformer-Example-NLU.png" alt="drawing" width="500"/>
</p>

*Could we move the CLS token to the start of the sentence?*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.h_dim = d_model // n_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask):
        # batch size B, sequence length L, d_model
        B, L, d_model = x.size() 

        q = self.w_q(x) # (B, L, d_model)
        k = self.w_k(x) # (B, L, d_model)
        v = self.w_v(x) # (B, L, d_model)

        # reshape to (B, n_heads, L, h_dim)
        q = q.view(B, L, self.n_heads, self.h_dim).transpose(1, 2)
        k = k.view(B, L, self.n_heads, self.h_dim).transpose(1, 2)
        v = v.view(B, L, self.n_heads, self.h_dim).transpose(1, 2)

        # dot-product between each head's q and k matrices
        # (B, n_heads, L, h_dim) * (B, n_heads, h_dim, L) -> (B, n_heads, L, L), i.e., similarities for each head
        similarity = q @ k.transpose(-2, -1) # transpose(-2,-1) transposes the k matrix for each head

        # normalize
        similarity = similarity * (1 / torch.sqrt(torch.tensor(self.h_dim)))

        # mask
        similarity = similarity.masked_fill(mask == 0, float('-inf'))

        attn = F.softmax(similarity, dim=-1)

        y = attn @ v  # (B, n_heads, L, L) * (B, n_heads, L, h_dim) -> (B, n_heads, L, h_dim)
        y = y.transpose(1, 2) # (B, L, n_heads, h_dim)
        # concatenate outputs of each head (contiguous is necessary to use view())
        y = y.contiguous().view(B, L, d_model) # (B, L, d_model)
        y = self.out_proj(y)

        return y

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, hidden_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, d_model),
        )

    def forward(self, x):
        # applied over each token independently
        return self.net(x)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, ff_dim, dropout)

    def forward(self, x, mask):
        # layer norm, attention, residual
        x = x + self.attn(self.ln1(x), mask)
        # layer norm, feedforward, residual
        x = x + self.ff(self.ln2(x))
        return x

In [ ]:
class GPT2(nn.Module):
    def __init__(
        self,
        vocab_size, 
        slots_size, 
        n_intents, 
        # GPT2 default hyperparameters
        pos_emb_size=1024,
        d_model=768,
        n_heads=12,
        num_layers=12,
        ff_dim=3072,
        dropout=0.1,
    ):
        super().__init__()
        self.pos_emb_size = pos_emb_size

        # learnable token embeddings
        self.token_embed = nn.Embedding(vocab_size, d_model)
        # learnable positional embeddings
        self.pos_embed = nn.Embedding(pos_emb_size, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)

        #### These are different from the Language Modeling case of Part 1
        # slot outputs (sequence labeling): one slot label for each input token
        self.slot_out = nn.Linear(d_model, slots_size)
        # intent output (sequence classification): classyfy the intent of the input sequence
        self.intent_out = nn.Linear(d_model, n_intents)

        # triangular matrix: (sub)token i will only be able to attend (sub)tokens j, 0 <= j <= i
        mask = torch.tril(torch.ones(pos_emb_size, pos_emb_size)).unsqueeze(0).unsqueeze(0)
        # mask is not a learnable parameter, 
        # but needs to be moved with everything else when doing .to(device) 
        self.register_buffer("mask", mask)
    
    def forward(self, idx, seq_lens):
        B, L = idx.shape # batch size, sequence length
        # positional embedding at most for position self.pos_emb_size
        # longer sequences cannot be processed (the model never learned positional embeddings for positions greater than self.pos_emb_size)
        assert L <= self.pos_emb_size

        pos = torch.arange(L, device=idx.device)
        x = self.token_embed(idx) + self.pos_embed(pos)

        mask = self.mask[:, :, :L, :L]

        for block in self.blocks:
            x = block(x, mask)

        # We are also predicting a slot for CLS
        # but it will be ignored, as we have a pad token in the ground truth
        x = self.ln_f(x)
        slots = self.slot_out(x)

        # get intent from last token (CLS)
        tmp = []
        # for sequence i in the batch, 
        # get the vector associated with its CLS token 
        # (last of the sequence)
        for i in range(x.shape[0]):
            tmp.append(x[i, seq_lens[i]-1]) # -1, we count from 0
        cls_tokens = torch.stack(tmp)
        # get logits for intents
        intent = self.intent_out(cls_tokens)

        return slots, intent # ((B, L, slot_size), (B, n_intents)

## 4.1 Function to randomly initialize the weights
This is a generic function that randomly initialize the parameters. 
To dig deep in to this you can look at: https://pytorch.org/docs/master/nn.init.html 


*Note: In Pytorch every parameter of the network has a proper name like weight_ih, weight_hh etc.*

In [ ]:
def init_weights(mat):
    for m in mat.modules():
        if type(m) in [nn.Linear]:
            torch.nn.init.uniform_(m.weight, -0.01, 0.01)
            if m.bias != None:
                m.bias.data.fill_(0.01)

## 4.2 Training set up

### Train Loop and Evaluation Loop
We define two functions one for training our model and the other for evaluating it. To compute the performances on the slot filling task we will use the **conll script**, while for the intent classification task we are going to use the **classification_report**.

<br>

In the literature, the Intent Classification task is evaluated using accuracy as a metric. The Slot filling task is evaluated using the *conll script* which computes the performance at the chunk level and the F1 score is usually reported. 

In [ ]:
from tqdm.notebook import tqdm
from conll import evaluate
from sklearn.metrics import classification_report

def train_loop(data, optimizer, criterion_slots, criterion_intents, model):
    model.train()
    loss_array = []

    for i, batch in enumerate(data):
        optimizer.zero_grad() # Zeroing the gradient

        slots, intent = model(batch['utterances'], batch['slots_len'])
        slots = slots.permute(0,2,1) # We need this for computing the loss

        loss_intent = criterion_intents(intent, batch['intents'])
        loss_slot = criterion_slots(slots, batch['y_slots'])
        loss = loss_intent + loss_slot # In joint training we sum the losses. 
                                       # Is there another way to do that?
        loss_array.append(loss.item())
        loss.backward() # Compute the gradient, deleting the computational graph
        optimizer.step() # Update the weights

    return loss_array

def eval_loop(data, criterion_slots, criterion_intents, model, lang):
    model.eval()
    loss_array = []
    
    ref_intents = []
    hyp_intents = []
    
    ref_slots = []
    hyp_slots = []
    with torch.no_grad(): # It used to avoid the creation of computational graph
        for batch in data:
            slots, intents = model(batch['utterances'], batch['slots_len'])
            slots = slots.permute(0,2,1) # We need this for computing the loss
            loss_intent = criterion_intents(intents, batch['intents'])
            loss_slot = criterion_slots(slots, batch['y_slots'])
            loss = loss_intent + loss_slot 
            loss_array.append(loss.item())

            # Intent inference
            # Get the most probable class
            out_intents = [lang.id2intent[x] for x in torch.argmax(intents, dim=1).tolist()] 
            gt_intents = [lang.id2intent[x] for x in batch['intents'].tolist()]
            ref_intents.extend(gt_intents)
            hyp_intents.extend(out_intents)
            
            # Slot inference 
            output_slots = torch.argmax(slots, dim=1)
            for id_seq, seq in enumerate(output_slots):
                length = batch['slots_len'].tolist()[id_seq] - 1 # -1, we ignore the CLS

                utt_ids = batch['utterances'][id_seq][:length].tolist()
                gt_ids = batch['y_slots'][id_seq][:length].tolist()
                gt_slots = [lang.id2slot[elem] for elem in gt_ids]
                utterance = [lang.id2word[elem] for elem in utt_ids]

                to_decode = seq[:length].tolist()
                ref_slots.append([(utterance[id_el], elem) for id_el, elem in enumerate(gt_slots)])
                tmp_seq = []
                for id_el, elem in enumerate(to_decode):
                    tmp_seq.append((utterance[id_el], lang.id2slot[elem]))
                hyp_slots.append(tmp_seq)
    try:            
        results = evaluate(ref_slots, hyp_slots)
    except Exception as ex:
        # Sometimes the model predicts a class that is not in REF
        print("Warning:", ex)
        ref_s = set([x[1] for x in ref_slots])
        hyp_s = set([x[1] for x in hyp_slots])
        print(hyp_s.difference(ref_s))
        results = {"total":{"f":0}}
        
    report_intent = classification_report(ref_intents, hyp_intents, 
                                          zero_division=False, output_dict=True)
    return results, report_intent, loss_array

### Model initialization
We initialize the model and we select the hyperparameters of the neural network. Futhermore, we initialize the optimizer and we select the loss function.
- You can find further optimization algorithms here: https://pytorch.org/docs/stable/optim.html
- and further loss functions here: https://pytorch.org/docs/stable/nn.html#loss-functions

In [ ]:
import torch.optim as optim

lr = 1 # This is definitely not good for AdamW

vocab_len = len(lang.word2id)
slots_len = len(lang.id2slot) # pad and cls have the same id
n_intents = len(lang.intent2id)

# Experiment also with a smaller or bigger model 
model = GPT2(
    vocab_len,
    slots_len,
    n_intents,
    pos_emb_size=1024,
    d_model=20,
    n_heads=1,
    num_layers=1,
    ff_dim=20,
).to(DEVICE)
model.apply(init_weights)

optimizer = optim.AdamW(model.parameters(), lr=lr)
criterion_slots = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)
criterion_intents = nn.CrossEntropyLoss() # No pad tokens, all sequences have a single label for intent

## 4.3 Train a neural network
We train a neural network iterating several times over the training set. 
* **epochs**: number of times in which the whole training set is seen by the network
* **early stopping**: keeps controlled the performance of the model on the dev set and interrupts the training when the performance is getting worse
    * **patience**: wait for a number of step before interrupting the training, even though the performance is getting worse. 

In [ ]:
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

n_epochs = 200
patience = 3
losses_train = []
losses_dev = []
sampled_epochs = []
best_f1 = 0
pbar = tqdm(range(n_epochs))
for i in pbar:
    loss = train_loop(train_loader, optimizer, criterion_slots, 
                      criterion_intents, model)
    if i % 1 == 0: # We check the performance every 1 epochs
        sampled_epochs.append(i)
        losses_train.append(np.asarray(loss).mean())
        results_dev, intent_res, loss_dev = eval_loop(dev_loader, criterion_slots, 
                                                      criterion_intents, model, lang)

        pbar.set_description(f"Slot F1: {results_dev['total']['f']:.2f}; Intent Acc: {intent_res['abbreviation']['f1-score']:.2f}")
        losses_dev.append(np.asarray(loss_dev).mean())
        
        f1 = results_dev['total']['f']
        # For decreasing the patience you can also use the average between slot f1 and intent accuracy
        if f1 > best_f1:
            best_f1 = f1
            # Here you should save the model
            patience = 3
        else:
            patience -= 1
        if patience <= 0: # Early stopping with patience
            break 

results_test, intent_test, _ = eval_loop(test_loader, criterion_slots, 
                                         criterion_intents, model, lang)    
print('Slot F1: ', results_test['total']['f'])
print('Intent Accuracy:', intent_test['accuracy'])

### Saving the model
To save the model you have to save:
- The weights of the model
- The computed vocabularies (w2id, slot2id, intent2id)
- The optimizer (optionally, only if you want to continue with the training)

In [ ]:
# PATH = os.path.join("bin", model_name)
# saving_object = {"epoch": x, 
#                  "model": model.state_dict(), 
#                  "optimizer": optimizer.state_dict(), 
#                  "w2id": w2id, 
#                  "slot2id": slot2id, 
#                  "intent2id": intent2id}
# torch.save(saving_object, PATH)

### Plot of the train and valid losses
One of the techniques for debugging a neural network is to check the plot of the loss. If the loss goes smoothly down then the network works correctly, otherwise a deeper analysis is needed. Furthermore, this plot can be useful for deciding the learning rate and the optimizer algorithm.

In [ ]:
plt.figure(num = 3, figsize=(8, 5)).patch.set_facecolor('white')
plt.title('Train and Dev Losses')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.plot(sampled_epochs, losses_train, label='Train loss')
plt.plot(sampled_epochs, losses_dev, label='Dev loss')
plt.legend()
plt.show()

### Multiple runs
To have reliable results on small corpora we have to train and test the model from scratch for several times. At the end, we average the results and we compute the standard deviation.

In [ ]:
lr = 1 # learning rate

vocab_len = len(lang.word2id)
slots_len = len(lang.id2slot) # pad and cls have the same id
n_intents = len(lang.intent2id)

n_epochs = 200
runs = 5

slot_f1s, intent_acc = [], []
for x in tqdm(range(0, runs)):
    model = GPT2(
        vocab_len,
        slots_len,
        n_intents,
        pos_emb_size=1024,
        d_model=20,
        n_heads=1,
        num_layers=1,
        ff_dim=20,
    ).to(DEVICE)
    model.apply(init_weights)

    optimizer = optim.AdamW(model.parameters(), lr=lr)
    criterion_slots = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)
    criterion_intents = nn.CrossEntropyLoss()

    patience = 3
    losses_train = []
    losses_dev = []
    sampled_epochs = []
    best_f1 = 0

    pbar = tqdm(range(n_epochs))
    for x in pbar:
        loss = train_loop(train_loader, optimizer, criterion_slots, 
                          criterion_intents, model)
        if x % 5 == 0:
            sampled_epochs.append(x)
            losses_train.append(np.asarray(loss).mean())
            results_dev, intent_res, loss_dev = eval_loop(dev_loader, criterion_slots, 
                                                          criterion_intents, model, lang)

            pbar.set_description(f"Slot F1: {results_dev['total']['f']:.2f}; Intent Acc: {intent_res['abbreviation']['f1-score']:.2f}")
            losses_dev.append(np.asarray(loss_dev).mean())

            f1 = results_dev['total']['f']

            if f1 > best_f1:
                best_f1 = f1
            else:
                patience -= 1
            if patience <= 0: # Early stopping with patient
                break # Not nice but it keeps the code clean

    results_test, intent_test, _ = eval_loop(test_loader, criterion_slots, 
                                             criterion_intents, model, lang)
    intent_acc.append(intent_test['accuracy'])
    slot_f1s.append(results_test['total']['f'])
slot_f1s = np.asarray(slot_f1s)
intent_acc = np.asarray(intent_acc)
print('Slot F1', round(slot_f1s.mean(),3), '+-', round(slot_f1s.std(),3))
print('Intent Acc', round(intent_acc.mean(), 3), '+-', round(slot_f1s.std(), 3))

 ![](https://huggingface.co/front/assets/huggingface_logo-noborder.svg)
# Hugging Face
Hugging Face is a library that allows you to use pretrained models in an easy way. This means that you do not need to implement an architecture and train it from scratch. Hugging Face is based on a community where people share trained models and code.
<br/><br/>
In Hugging Face there are many different models (https://huggingface.co/models) that you can import and each of them has its own input and output shapes. However, Transformer-based models are usually composed of two parts: 
- **Tokenizer**
- **Architecture/Pretrained model**

In [ ]:
from transformers import AutoModel, AutoTokenizer
from pprint import pprint

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token
# not good parameters for alpha and rank
model = AutoModel.from_pretrained("openai-community/gpt2")

inputs = tokenizer(["I saw a man with a telescope", "StarLord was here",  "I didn't"], return_tensors="pt", padding=True)
pprint(inputs)

outputs = model(**inputs)

last_hidden_states = outputs.last_hidden_state
print(last_hidden_states.shape)

print(inputs["input_ids"][0])
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][1]))

In [ ]:
# BERT model script from: huggingface.co

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased") # Download the tokenizer
model = AutoModel.from_pretrained("bert-base-uncased") # Download the model

inputs = tokenizer(["I saw a man with a telescope", "StarLord was here",  "I didn't"], return_tensors="pt", padding=True)
pprint(inputs)

outputs = model(**inputs)

last_hidden_states = outputs.last_hidden_state
print(last_hidden_states.shape)


print(inputs["input_ids"][0])
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][1]))

##  Byte pair encoding
The tricky part of using Transformer-based model is the tokenizer which is based on the byte pair encoding algorithm. Indeed, the **tokenizers** used by Transformer-based models are different from what we used in this lab. While for instance Spacy's tokenizer is rule-based and splits the text looking at the punctuation, the goal of Transformer tokenizers is to reduce the vocabulary length by splitting words into subwords. A thorough explanation of these tokenizers can be found here: https://huggingface.co/docs/transformers/tokenizer_summary  

# Mandatory Exam Exercise

## Part 2.A
As for the first part of the project (LM), you have to apply modifications incrementally. Also in this case you may have to play with the hyperparameters and optimizers to improve the performance. 

Modify the baseline architecture GPT2:

0. Baseline: find a suitable learning rate, but keep the other hyperparameters fixed.
1. Hyperparameter optimization: incrementally change hyperparameters one at a time (```d_model```, ```n_heads```, ```num_layers```, ```ff_dim```). You may have to readjust the learning rate for larger model sizes.
2. Adding a dropout layer before the final output layers

**Intent classification**: accuracy <br>
**Slot filling**: F1 score with conll

***Dataset to use: ATIS***

## Implementation Part 2.A


In [ ]:
# Part 2.A - experiments runner
import copy
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter

# Re-implement GPT2 for intent/slot with optional output dropout
class GPT2_Mod(nn.Module):
    def __init__(
        self,
        vocab_size,
        slots_size,
        n_intents,
        pos_emb_size=1024,
        d_model=768,
        n_heads=12,
        num_layers=12,
        ff_dim=3072,
        dropout=0.1,
        out_dropout=0.0,
    ):
        super().__init__()
        self.pos_emb_size = pos_emb_size

        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(pos_emb_size, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.out_dropout = nn.Dropout(out_dropout)

        # slot outputs (sequence labeling)
        self.slot_out = nn.Linear(d_model, slots_size)
        # intent output (sequence classification)
        self.intent_out = nn.Linear(d_model, n_intents)

        mask = torch.tril(torch.ones(pos_emb_size, pos_emb_size)).unsqueeze(0).unsqueeze(0)
        self.register_buffer("mask", mask)
    
    def forward(self, idx, seq_lens):
        B, L = idx.shape
        assert L <= self.pos_emb_size

        pos = torch.arange(L, device=idx.device)
        x = self.token_embed(idx) + self.pos_embed(pos)
        mask = self.mask[:, :, :L, :L]
        for block in self.blocks:
            x = block(x, mask)

        x = self.ln_f(x)
        x = self.out_dropout(x)
        slots = self.slot_out(x)

        tmp = []
        for i in range(B):
            tmp.append(x[i, seq_lens[i] - 1])
        cls_tokens = torch.stack(tmp)
        intent = self.intent_out(cls_tokens)
        return slots, intent

@dataclass
class ExperimentConfig:
    name: str
    d_model: int = 20
    n_heads: int = 1
    num_layers: int = 1
    ff_dim: int = 20
    lr: float = 1e-3
    dropout: float = 0.1
    out_dropout: float = 0.0
    n_epochs: int = 50
    patience: int = 3

def _make_run_dir(name: str) -> Path:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_name = name.replace(" ", "_")
    run_dir = Path("runs") / f"{timestamp}_{safe_name}"
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir

def run_experiment(cfg: ExperimentConfig):
    print(f"\n=== {cfg.name} ===")
    torch.manual_seed(42)
    model = GPT2_Mod(
        vocab_len,
        slots_len,
        n_intents,
        pos_emb_size=1024,
        d_model=cfg.d_model,
        n_heads=cfg.n_heads,
        num_layers=cfg.num_layers,
        ff_dim=cfg.ff_dim,
        dropout=cfg.dropout,
        out_dropout=cfg.out_dropout,
    ).to(DEVICE)
    model.apply(init_weights)

    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr)
    criterion_slots = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)
    criterion_intents = nn.CrossEntropyLoss()

    run_dir = _make_run_dir(cfg.name)
    writer = SummaryWriter(log_dir=str(run_dir / "tb"))
    checkpoint_path = run_dir / f"checkpoint_{cfg.name.replace(' ', '_')}.pt"

    best_f1 = 0.0
    best_model = copy.deepcopy(model).to("cpu")
    patience = cfg.patience
    for epoch in tqdm(range(cfg.n_epochs), desc="Epochs"):
        loss = train_loop(train_loader, optimizer, criterion_slots, criterion_intents, model)
        results_dev, intent_res, _ = eval_loop(dev_loader, criterion_slots, criterion_intents, model, lang)
        f1 = results_dev["total"]["f"]
        intent_acc = intent_res["accuracy"]
        writer.add_scalar("loss/train", float(sum(loss) / max(len(loss), 1)), epoch)
        writer.add_scalar("slot_f1/dev", f1, epoch)
        writer.add_scalar("intent_acc/dev", intent_acc, epoch)
        if f1 > best_f1:
            best_f1 = f1
            best_model = copy.deepcopy(model).to("cpu")
            torch.save(best_model.state_dict(), checkpoint_path)
            patience = cfg.patience
        else:
            patience -= 1
        if patience <= 0:
            break
        print(f"Epoch {epoch}: Slot F1={f1:.4f} | Intent Acc={intent_acc:.4f}")

    best_model.to(DEVICE)
    results_test, intent_test, _ = eval_loop(test_loader, criterion_slots, criterion_intents, best_model, lang)
    if "total" in results_test:
        writer.add_scalar("slot_f1/test", results_test["total"]["f"])
    writer.add_scalar("intent_acc/test", intent_test["accuracy"])
    writer.close()
    print(f"Best dev Slot F1: {best_f1:.4f} | Test Slot F1: {results_test['total']['f']:.4f} | Test Intent Acc: {intent_test['accuracy']:.4f}")
    return best_f1, results_test["total"]["f"], intent_test["accuracy"]

# === Experiments (edit these as you test) ===
experiments = [
    # 0) Baseline: tune lr only, keep hyperparameters fixed
    ExperimentConfig(name="Baseline", lr=1e-3),
    # 1) Hyperparameter optimization (one change at a time)
    ExperimentConfig(name="Bigger d_model", d_model=32, ff_dim=64, lr=5e-4),
    ExperimentConfig(name="More heads", n_heads=2, d_model=32, ff_dim=64, lr=5e-4),
    ExperimentConfig(name="More layers", num_layers=2, d_model=32, ff_dim=64, lr=3e-4),
    # 2) Add dropout before final output layers
    ExperimentConfig(name="+ out dropout", d_model=32, ff_dim=64, out_dropout=0.1, lr=3e-4),
]

# strict type validation
if not isinstance(experiments, list) or not all(isinstance(x, ExperimentConfig) for x in experiments):
    raise TypeError(f"experiments must be a list of ExperimentConfig, got {[type(x) for x in experiments]}")

results = []
for cfg in experiments:
    results.append((cfg.name, *run_experiment(cfg)))

print("\nSummary (name, best dev Slot F1, test Slot F1, test Intent Acc):")
for r in results:
    print(r)

## Part 2.B

Adapt the code to fine-tune a pre-trained GPT2 (decoder only) and BERT (encoder only) models using a multi-task learning setting on intent classification and slot filling. 

For both models, one of the challenges of this is to **handle the sub-tokenization issue**.  You can refer to this paper to have a better understanding of how to implement this (they use BERT): https://arxiv.org/abs/1902.10909. 

As the models have different architectures, you will have to perform intent classification differently. 

*Note*: The fine-tuning process is to further train on a specific task/s a model that has been pre-trained on a different (potentially unrelated) task/s.


For GPT2, you can experiment with [*GPT2* or *GPT2-medium*](https://huggingface.co/openai-community/gpt2). For BERT, you can experiment with [*BERT-base* or *BERT-large*](https://huggingface.co/google-bert/bert-base-uncased). 

**Intent classification**: accuracy <br>
**Slot filling**: F1 score with conll

***Dataset to use: ATIS***

# Multi-Task Learning with BERT and GPT-2 (Intent + Slot Filling)

## Task Definition

Input: sentence
Outputs:

* Intent (single label)
* Slots (one label per word)

Dataset: ATIS

---

# 1. Common Pipeline (Shared Logic)

```
Text
  ↓
Tokenization (subwords)
  ↓
Label alignment
  ↓
Transformer (BERT or GPT-2)
  ↓
Hidden states (per token)
  ↓
  ├── Intent head (sentence-level)
  └── Slot head (token-level)
  ↓
Loss computation
```

---

# 2. Sub-token Label Alignment

Problem:

```
1 word → multiple tokens
```

Solution:

* Assign label to **first sub-token**
* Assign `-100` to remaining sub-tokens

Example:

```
Word:   milan
Tokens: mi   ##lan
Labels: B-from_city   -100
```

Loss:

```
CrossEntropy(ignore_index = -100)
```

---

# 3. BERT Architecture (Encoder)

Model: BERT (bidirectional)

## Pipeline

```
Input text
  ↓
WordPiece tokenization
  ↓
Label alignment
  ↓
BERT encoder
  ↓
Hidden states (N tokens × hidden_dim)
  ↓
  ├── [CLS] token
  │       ↓
  │   Linear → intent logits
  │
  └── All token vectors
          ↓
      Linear → slot logits (per token)
```

## Key Points

* `[CLS]` is the **first token**
* `[CLS]` attends to all tokens → sentence representation
* Slot predictions use all token embeddings

## Loss

```
intent_loss = CrossEntropy(intent_logits, intent_label)
slot_loss   = CrossEntropy(slot_logits, slot_labels)

total_loss = intent_loss + slot_loss
```

---

# 4. GPT-2 Architecture (Decoder)

Model: GPT-2 (causal / left-to-right)

## Pipeline

```
Input text
  ↓
BPE tokenization
  ↓
Label alignment
  ↓
GPT-2 decoder (causal attention)
  ↓
Hidden states (N tokens × hidden_dim)
  ↓
  ├── Sentence representation
  │       ↓
  │   (last token OR mean pooling)
  │       ↓
  │   Linear → intent logits
  │
  └── All token vectors
          ↓
      Linear → slot logits (per token)
```

## Sentence Representation Options

Option A (default):

```
last_token = hidden_states[:, -1]
```

Option B:

```
mean_pool = mean(hidden_states)
```

## Key Points

* No `[CLS]` token
* Last token sees full context (causal attention)
* Earlier tokens do NOT see future tokens

---

# 5. Slot Filling Head (Both Models)

```
hidden_states → Linear → slot_logits
```

* Shape:

```
(batch_size, seq_len, num_slot_labels)
```

* Loss ignores sub-tokens (`-100`)

---

# 6. Intent Head (Both Models)

BERT:

```
hidden_states[:, 0] → Linear → intent
```

GPT-2:

```
hidden_states[:, -1] OR mean → Linear → intent
```

---

# 7. Training Objective

```
total_loss = intent_loss + slot_loss
```

Optional weighting:

```
total_loss = α * intent_loss + β * slot_loss
```

Default:

```
α = 1, β = 1
```

---

# 8. Inference

Slots:

* Use predictions from **first sub-token only**

Intent:

* Direct classifier output

---

# 9. Key Differences

| Component    | BERT          | GPT-2             |
| ------------ | ------------- | ----------------- |
| Architecture | Encoder       | Decoder           |
| Context      | Bidirectional | Left-to-right     |
| Intent input | `[CLS]`       | Last token / mean |
| Tokenization | WordPiece     | BPE               |
| Slot context | Full          | Partial (Past tokens)|

---

# 10. Minimal Implementation Structure

## Forward Pass

```
inputs → transformer → hidden_states

intent_logits = intent_head(...)
slot_logits   = slot_head(...)

return {
  loss,
  intent_logits,
  slot_logits
}
```

---

# 11. Summary

```
One transformer backbone
+ two heads (intent + slots)
+ aligned labels
+ summed loss
```
